In [ ]:
import sympy as sp
import numpy as np
import matplotlib.pyplot as plt
import ipywidgets as widgets
from IPython.display import display, clear_output, HTML

display(HTML("<style>.output_scroll { height: unset !important; }</style>"))

def symbolic_dtft_from_scratch():
    n, N, omega = sp.symbols('n N omega', real=True, integer=True)
    alpha = sp.symbols('alpha', positive=True, real=True)

    # --- 1. SYMBOLIC DTFT FROM FIRST PRINCIPLES ---

    r_pos = alpha * sp.exp(-sp.I * omega)
    finite_pos = (1 - r_pos**(N + 1)) / (1 - r_pos)
    sum_pos_evaluated = sp.simplify(finite_pos.subs(r_pos**(N + 1), 0))

    r_neg = alpha * sp.exp(sp.I * omega)
    finite_neg = r_neg * (1 - r_neg**N) / (1 - r_neg)
    sum_neg_evaluated = sp.simplify(finite_neg.subs(r_neg**N, 0))

    X_omega_sym = sp.simplify(sum_pos_evaluated + sum_neg_evaluated)
    X_omega_simplified = sp.trigsimp(sp.factor(sp.together(sp.expand_complex(X_omega_sym))))
    magnitude_sym = sp.simplify(sp.Abs(X_omega_simplified))

    print(r"--- Symbolically Derived Results ---")
    display(HTML(r"<b>Sum ($n \ge 0$) =</b> $" + sp.latex(sum_pos_evaluated) + "$"))
    display(HTML(r"<b>Sum ($n < 0$) =</b> $" + sp.latex(sum_neg_evaluated) + "$"))
    display(HTML(r"<b>DTFT $X(e^{j\omega}) =$</b> $" + sp.latex(X_omega_simplified) + "$"))
    display(HTML(r"<b>Magnitude $|X(e^{j\omega})| =$</b> $" + sp.latex(magnitude_sym) + "$"))

    f_mag = sp.lambdify((omega, alpha), magnitude_sym, modules='numpy')

    def update_plots(alpha_val):
        clear_output(wait=True)
        omega_vals = np.linspace(-3*np.pi, 3*np.pi, 4000)
        mag_vals = np.real(f_mag(omega_vals, alpha_val))

        fig, ax = plt.subplots(figsize=(12, 5.5))
        
        # Στρογγυλοποίηση στα 3 δεκαδικά ψηφία και τοποθέτηση υπομνήματος έξω δεξιά
        ax.plot(omega_vals/np.pi, mag_vals, color='red', lw=2.5, label=f'Magnitude, alpha = {alpha_val:.3f}')
        ax.set_title(f'DTFT Magnitude Spectrum of x[n] = alpha^|n|    (alpha = {alpha_val:.3f})', fontsize=12, fontweight='bold')
        ax.set_xlabel('Normalized Frequency omega / pi', fontsize=10)
        ax.set_ylabel('Magnitude', fontsize=10)
        ax.set_xlim(-3, 3)
        ax.set_xticks([-3, -2, -1, 0, 1, 2, 3])
        ax.set_xticklabels(['-3π', '-2π', '-π', '0', 'π', '2π', '3π'])
        ax.grid(True, linestyle='--', alpha=0.6)
        
        # Μετακίνηση υπομνήματος έξω από το σχήμα (στα δεξιά)
        ax.legend(loc='center left', bbox_to_anchor=(1.02, 0.5), fontsize=9)
        
        # Προσαρμογή περιθωρίων για να χωράει το legend
        plt.subplots_adjust(left=0.08, right=0.80, top=0.90, bottom=0.12)
        plt.show()

        display(HTML("""
        <div style="border:1px solid #ccc; padding:12px; border-radius:5px; background-color:#f9f9f9; font-family:sans-serif; font-size:14px;">
        <strong>Symbolic DTFT:</strong> The spectrum was obtained from the symbolic evaluation of the two infinite sums defining the DTFT. The plotted expression is the resulting symbolic expression evaluated numerically for the selected value of alpha.
        </div>
        """))

    alpha_slider = widgets.FloatSlider(value=0.5, min=0.05, max=0.95, step=0.005, description='Parameter alpha:', style={'description_width':'initial'})
    ui = widgets.VBox([alpha_slider])
    out = widgets.interactive_output(update_plots, {'alpha_val': alpha_slider})
    display(ui, out)

symbolic_dtft_from_scratch()